# Trash Classification Pipeline

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.inference.predictor import RecyclingPredictor
from src.inference.camera import CameraInference
from src.utils.config import Config

## 1. Load Predictor

In [2]:
import serial
import time

class ArduinoController:
    """
    Drives the Arduino Nano by sending classification instructions (0-3)
    over the USB serial link.
    """
    def __init__(self, port='COM6', baudrate=9600, timeout=2):
        """
        Initialise the connection to the Arduino.
        
        Args:
            port: Serial port (e.g. 'COM3' on Windows, '/dev/ttyUSB0' on Linux)
            baudrate: Communication speed (Arduino default is 9600)
            timeout: Read timeout in seconds
        """
        try:
            self.ser = serial.Serial(port, baudrate, timeout=timeout)
            time.sleep(2)  # Wait for the Arduino to reset after connecting
            self.ser.reset_input_buffer()
            print(f"✓ Connected to Arduino on port {port} at {baudrate} baud")
            
            # Read the Arduino startup message
            if self.ser.in_waiting:
                msg = self.ser.readline().decode('utf-8', errors='ignore').strip()
                print(f"  Arduino: {msg}")
        except serial.SerialException as e:
            print(f"✗ Error connecting to Arduino: {e}")
            print("  Check that the Arduino is connected and the port is correct")
            self.ser = None
    
    def send_instruction(self, class_id):
        """
        Send a classification instruction to the Arduino.
        
        Args:
            class_id: Class identifier (0-3)
                - 0: cardboard_paper
                - 1: ecoglasses
                - 2: metal_plastic
                - 3: trash
        
        Returns:
            bool: True if the instruction was sent, False on error
        """
        if self.ser is None or not self.ser.is_open:
            print("✗ Arduino not connected")
            return False
        
        if not isinstance(class_id, int) or class_id < 0 or class_id > 3:
            print(f"✗ Invalid class ID: {class_id}. Must be between 0 and 3")
            return False
        
        try:
            # Send as an ASCII character ('0'-'3')
            self.ser.write(str(class_id).encode('utf-8'))
            print(f"→ Sent: {class_id}")
            
            # Read the Arduino response
            response = self.ser.readline().decode('utf-8', errors='ignore').strip()
            if response:
                print(f"  Arduino: {response}")
            
            return True
        except Exception as e:
            print(f"✗ Error sending instruction: {e}")
            return False
    
    def close(self):
        """Close the Arduino connection."""
        if self.ser and self.ser.is_open:
            self.ser.close()
            print("✓ Connection closed")

# Attempt to connect - CHANGE THE PORT IF NEEDED
# En Windows: 'COM3', 'COM4', etc.
# En Linux: '/dev/ttyUSB0', '/dev/ttyACM0', etc.
# En Mac: '/dev/tty.usbserial-*'
try:
    arduino = ArduinoController(baudrate=9600)
except:
    print("Note: adjust the serial port for your system")
    arduino = None

✓ Connected to Arduino on port COM6 at 9600 baud


## Arduino Serial Communication Setup

In [3]:
config = Config('../configs/mobilenet_config.yaml')
model_path = '../outputs/checkpoints/best_model.pt'

class_mapping = {0: 'cardboard_paper', 1: 'ecoglasses', 2: 'metal_plastic', 3: 'trash'}

predictor = RecyclingPredictor(
    model_path=model_path,
    num_classes=config.model.get('num_classes'),
    architecture=config.model.get('architecture'),
    class_mapping=class_mapping
)
print("Predictor loaded.")

Predictor loaded.


C:\Users\tomas\OneDrive\Documents\GitHub\TP-final-vision\src\data\augmentation.py:67: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(


In [8]:
from src.inference.camera_gradcam import CameraInferenceWithGradCAM

class CameraInferenceWithArduino(CameraInferenceWithGradCAM):
    """Extends CameraInferenceWithGradCAM to forward instructions to the Arduino."""
    
    def __init__(self, predictor, arduino_controller, camera_id=0, enable_gradcam=True, 
                 stability_duration=4.0, stereo_mode=None, width=1280, height=1280,
                 black_threshold=0.7, brightness_threshold=40):
        super().__init__(predictor, camera_id=camera_id, enable_gradcam=enable_gradcam, 
                        stability_duration=stability_duration, stereo_mode=stereo_mode, 
                        width=width, height=height,
                        black_threshold=black_threshold, brightness_threshold=brightness_threshold)
        self.arduino = arduino_controller
    
    def process_frame(self, result):
        """Override process_frame to send to the Arduino once a class is stable."""
        if 'stable_class' in result:
            stable_class = result['stable_class']
            class_name = self.predictor.class_mapping.get(stable_class, f"Class {stable_class}")
            
            print(f"\n🎯 STABLE classification: {class_name} (ID: {stable_class})")
            
            if self.arduino and self.arduino.ser:
                success = self.arduino.send_instruction(stable_class)
                if success:
                    self.last_sent_class = stable_class
                    print(f"✅ Sent to Arduino")
            else:
                print(f"⚠️  Arduino not connected")
                self.last_sent_class = stable_class

# 🔧 CONFIGURATION
STEREO_MODE = 'left'  # 'left' or 'right' for stereo, None for mono
BLACK_THRESHOLD = 0.6  # share of dark pixels above which the tray counts as empty
BRIGHTNESS_THRESHOLD = 50  # Maximum brightness for a pixel to count as "dark"

if arduino and arduino.ser:
    print(f"🚀 Starting camera with Arduino")
    print(f"📷 Mode: {STEREO_MODE or 'MONO'}")
    print(f"🎯 Empty detection: {BLACK_THRESHOLD*100:.0f}% dark, brightness < {BRIGHTNESS_THRESHOLD}")
    
    camera_arduino = CameraInferenceWithArduino(
        predictor, arduino, camera_id=0, enable_gradcam=True,
        stability_duration=4.0, stereo_mode=STEREO_MODE, 
        width=1800, height=1800,
        black_threshold=BLACK_THRESHOLD,
        brightness_threshold=BRIGHTNESS_THRESHOLD
    )
    camera_arduino.run()
else:
    print("⚠ Arduino not available")
    camera = CameraInferenceWithGradCAM(
        predictor, camera_id=0, enable_gradcam=True, 
        stability_duration=4.0, stereo_mode=STEREO_MODE,
        black_threshold=BLACK_THRESHOLD, width=1800, height=1800,
        brightness_threshold=BRIGHTNESS_THRESHOLD
    )
    camera.run()

🚀 Starting camera with Arduino
📷 Mode: left
🎯 Empty detection: 60% dark, brightness < 50

🎯 STABLE classification: ecoglasses (ID: 1)
→ Sent: 1
  Arduino: Done.
✅ Sent to Arduino


**Tunable parameters:**
- `black_threshold`: Share of dark pixels required to consider the tray empty (default: 0.7 = 70%)
- `brightness_threshold`: Maximum brightness for a pixel to count as "dark" (default: 40 on a 0-255 scale)

In [5]:
# Fallback examples: alternative empty-tray detection settings to try

# MORE SENSITIVE configuration (registers as empty more easily)
# Use if the system classifies the dark background as trash
# BLACK_THRESHOLD = 0.6  # 60% dark pixels
# BRIGHTNESS_THRESHOLD = 50  # More tolerant of brightness

# LESS SENSITIVE configuration (harder to register as empty)
# Use if it reports empty while an object is present
# BLACK_THRESHOLD = 0.8  # 80% dark pixels
# BRIGHTNESS_THRESHOLD = 30  # Stricter on brightness

# STANDARD configuration (recommended starting point)
#BLACK_THRESHOLD = 0.7  # 70% dark pixels
#BRIGHTNESS_THRESHOLD = 40  # Balance medio

In [6]:
#arduino.close()